In [6]:
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

SEED = 42

# Fix random seeds to make experiments more reproducible.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Normal device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set paths for files
project_dir = Path.cwd()
data_dir = project_dir / "data" / "keras_slices_data"
checkpoint_dir = project_dir / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

# Setup constants
NUM_CLASSES = 6
PROSTATE_LABEL = 5 # Other labels can be found in dataset notebook
BATCH_SIZE = 16
# Value chosen after initial training split showed progress slowed after around 20 epochs.
# Also lines up with ~1h of training time for training restriction (A100 GPU)
EPOCHS = 20 

LEARNING_RATE = 1e-3

print("Device:", device)
print("Dataset:", data_dir)
print("Prostate label:", PROSTATE_LABEL)

FileNotFoundError: [Errno 2] No such file or directory: '/home/chris/recognition/checkpoints'

In [ ]:
# Run dataset definitions and checks in this notebook's kernel.
%run ./dataset.ipynb

# Run the baseline architecture definitions.
%run ./modules.ipynb

In [ ]:
# Re-establish settings after running the dataset and modules notebooks
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

data_dir = project_dir / "data" / "keras_slices_data"
splits = audit_splits(data_dir)

# Data loaders
# Shuffle only training data
train_loader = DataLoader(
    HipMRIDataset(splits["train"]),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    HipMRIDataset(splits["validate"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

# Create fresh baseline
model = UNet(classes=NUM_CLASSES, base=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Prostate class:", PROSTATE_LABEL)

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

# Ignore artificial padding added by the dataset loader.
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)


def run_epoch(model, loader, training=False):
    model.train(training)
    loss_sum, pixel_count = 0.0, 0
    intersection, predicted_count, target_count = 0, 0, 0

    # Limit progress-bar updates to avoid Jupyter message-rate warnings.
    for images, masks in tqdm(loader, leave=False, mininterval=5):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        # Validation calculates predictions without storing gradients.
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, masks)

            if not torch.isfinite(loss).item():
                raise RuntimeError("Non-finite loss encountered")

            if training:
                loss.backward()
                optimizer.step()

        # Weight average loss by real pixels, excluding padding.
        valid = masks != -100
        pixels = valid.sum().item()
        loss_sum += loss.item() * pixels
        pixel_count += pixels

        # Accumulate prostate overlap across the entire epoch.
        prediction = logits.detach().argmax(dim=1)
        pred_prostate = (prediction == PROSTATE_LABEL) & valid
        true_prostate = (masks == PROSTATE_LABEL) & valid

        intersection += (pred_prostate & true_prostate).sum().item()
        predicted_count += pred_prostate.sum().item()
        target_count += true_prostate.sum().item()

    denominator = predicted_count + target_count
    dice = (
        2 * intersection / denominator
        if denominator > 0 else float("nan")
    )

    return loss_sum / pixel_count, dice

In [ ]:
# Train the baseline
history = []
best_dice = -1.0

checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "baseline_best.pt"
history_path = checkpoint_dir / "baseline_history.csv"

# Start the clock measuring time budget for the training
TIME_LIMIT_SECONDS = 60 * 60
training_start = time.perf_counter()

# Track peak PyTorch GPU memory during training and validation.
if device.type == "cuda":
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)

for epoch in range(1, EPOCHS + 1):
    start = time.perf_counter()

    train_loss, train_dice = run_epoch(
        model, train_loader, training=True
    )
    val_loss, val_dice = run_epoch(
        model, val_loader, training=False
    )

    seconds = time.perf_counter() - start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_prostate_dice": train_dice,
        "val_prostate_dice": val_dice,
        "seconds": seconds,
    })

    # Save the history before checking whether to stop.
    pd.DataFrame(history).to_csv(history_path, index=False)

    if val_dice > best_dice:
        best_dice = val_dice

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_prostate_dice": val_dice,
            "num_classes": NUM_CLASSES,
            "prostate_label": PROSTATE_LABEL,
            "base": 16,
            "target_size": (256, 144),
            "model_type": "baseline",
        }, checkpoint_path)

    elapsed = time.perf_counter() - training_start

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss {train_loss:.4f} / {val_loss:.4f} | "
        f"prostate Dice {train_dice:.4f} / {val_dice:.4f} | "
        f"{seconds:.1f}s | best val {best_dice:.4f} | "
        f"elapsed {elapsed / 60:.1f} min",
        flush=True,
    )

    # Stop cleanly after finishing and recording the current epoch.
    if elapsed >= TIME_LIMIT_SECONDS:
        print("One-hour training limit reached. Stopping.")
        break

if device.type == "cuda":
    print(
        "Training/validation peak allocated VRAM:",
        f"{torch.cuda.max_memory_allocated(device) / 1024**3:.3f} GiB"
    )
    print(
        "Training/validation peak reserved VRAM:",
        f"{torch.cuda.max_memory_reserved(device) / 1024**3:.3f} GiB"
    )

print("Best checkpoint:", checkpoint_path)

In [ ]:
# Epoch 20/20 | loss 0.0390 / 0.0761 | prostate Dice 0.8941 / 0.8304 | 128.6s | best val 0.8304 | elapsed 43.1 min
# Training/validation peak allocated VRAM: 0.727 GiB
#Training/validation peak reserved VRAM: 0.906 GiB
#Best checkpoint: /lambda/nfs/comp3710-virginia/PatternAnalysis-2026/recognition/ResidualUNet_s4749642/checkpoints/baseline_best.pt

In [ ]:
# Load the best saved weights before measuring inference.
checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Select 100 validation slices spread across the validation set.
indices = np.linspace(
    0, len(val_loader.dataset) - 1,
    num=min(100, len(val_loader.dataset)),
    dtype=int,
)

# Preload inputs so timing excludes disk access and preprocessing.
# Batch size 1 measures latency for one 2D slice.
inputs = [
    val_loader.dataset[int(i)][0].unsqueeze(0)
    for i in indices
]


def synchronise():
    if device.type == "cuda":
        torch.cuda.synchronize(device)


times_ms = []

with torch.inference_mode():
    # Warm up the GPU before recording timings.
    warmup = inputs[0].to(device)
    for _ in range(10):
        _ = model(warmup).argmax(dim=1)

    synchronise()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for image in inputs:
        # Transfer before starting the timer: GPU inference only.
        image = image.to(device)
        synchronise()
        start = time.perf_counter()

        prediction = model(image).argmax(dim=1)

        synchronise()
        times_ms.append((time.perf_counter() - start) * 1000)

    # Release the final prediction and input references.
    del prediction, image, warmup

results = {
    "Model": checkpoint["model_type"],
    "Checkpoint epoch": checkpoint["epoch"],
    "Parameters": sum(p.numel() for p in model.parameters()),
    "Inference batch size": 1,
    "Slices measured": len(times_ms),
    "Mean inference time (ms/slice)": np.mean(times_ms),
    "Median inference time (ms/slice)": np.median(times_ms),
    "95th percentile time (ms/slice)": np.percentile(times_ms, 95),
}

if device.type == "cuda":
    results["Peak inference allocated VRAM (GiB)"] = (
        torch.cuda.max_memory_allocated(device) / 1024**3
    )
    results["Peak inference reserved VRAM (GiB)"] = (
        torch.cuda.max_memory_reserved(device) / 1024**3
    )

profile = pd.Series(results, name="Resource profile").to_frame()
display(profile)

# Keep measurements separately from the model weights.
profile.to_csv(checkpoint_dir / "baseline_resource_profile.csv")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    checkpoint_dir / "baseline_best.pt",
    map_location=device,
    weights_only=False,  # Load only your own trusted checkpoint.
)

preview_model = UNet(
    classes=checkpoint["num_classes"],
    base=checkpoint["base"],
).to(device)

preview_model.load_state_dict(checkpoint["model_state_dict"])
preview_model.eval()

print("Loaded best baseline from epoch:", checkpoint["epoch"])
print("Validation prostate Dice:", checkpoint["val_prostate_dice"])

In [ ]:
import matplotlib.pyplot as plt

PATIENT_ID = "036"  # Choose 036, 037, 038 or 039
WEEK = 0

# Select this patient's scan and sort slices numerically.
case_pairs = [
    pair for pair in splits["validate"]
    if pair[0].name.startswith(
        f"case_{PATIENT_ID}_week_{WEEK}_slice_"
    )
]
case_pairs.sort(
    key=lambda pair: int(pair[0].name.split("_slice_")[1].split(".")[0])
)

selected = case_pairs[::5]
assert selected, "No matching slices found."

# Reuse the dataset preprocessing, including padding.
preview_dataset = HipMRIDataset(selected)

fig, axes = plt.subplots(
    len(selected), 4,
    figsize=(12, 3 * len(selected)),
    squeeze=False,
)
colours = dict(cmap="tab10", vmin=0, vmax=9, interpolation="nearest")

preview_model.eval()

for row, (image_path, _) in enumerate(selected):
    image, mask = preview_dataset[row]
    slice_number = image_path.name.split("_slice_")[1].split(".")[0]

    with torch.no_grad():
        prediction = preview_model(image.unsqueeze(0).to(device))
        prediction = prediction.argmax(dim=1)[0].cpu().numpy()

    image = image[0].numpy()
    mask = mask.numpy()
    valid = mask != -100

    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 1].imshow(np.ma.masked_where(~valid, mask), **colours)
    axes[row, 2].imshow(
        np.ma.masked_where(~valid, prediction), **colours
    )

    axes[row, 3].imshow(image, cmap="gray")
    axes[row, 3].imshow(
        np.ma.masked_where((prediction == 0) | ~valid, prediction),
        alpha=0.4,
        **colours,
    )

    for ax, title in zip(
        axes[row],
        ["MRI", "Ground truth", "Prediction", "Prediction overlay"],
    ):
        ax.set_title(f"Slice {slice_number}: {title}")
        ax.axis("off")

fig.suptitle(f"Patient {PATIENT_ID}, week {WEEK}: every fifth slice")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt

# Calculate symmetric HD95 between the boundaries of two binary masks.
def calculate_hd95(predicted, reference):
    if not predicted.any() and not reference.any():
        return np.nan  # No structure present: boundary distance is undefined.

    if not predicted.any() or not reference.any():
        return np.inf  # A structure was completely missed or falsely predicted.

    predicted_boundary = predicted & ~binary_erosion(
        predicted, border_value=0
    )
    reference_boundary = reference & ~binary_erosion(
        reference, border_value=0
    )

    # Distance from every pixel to the closest boundary pixel.
    distance_to_reference = distance_transform_edt(~reference_boundary)
    distance_to_prediction = distance_transform_edt(~predicted_boundary)

    distances = np.concatenate([
        distance_to_reference[predicted_boundary],
        distance_to_prediction[reference_boundary],
    ])
    return float(np.percentile(distances, 95))


# Accumulate overlap counts across all validation slices.
intersections = np.zeros(NUM_CLASSES, dtype=np.int64)
predicted_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
reference_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
boundary_distances = [[] for _ in range(NUM_CLASSES)]

preview_model.eval()

with torch.inference_mode():
    for images, masks in tqdm(
        val_loader, desc="Validation metrics", mininterval=5
    ):
        predictions = (
            preview_model(images.to(device))
            .argmax(dim=1)
            .cpu()
            .numpy()
        )
        masks = masks.numpy()

        for prediction, reference in zip(predictions, masks):
            valid = reference != -100

            # Remove padding before extracting boundaries.
            rows, columns = np.where(valid)
            region = (
                slice(rows.min(), rows.max() + 1),
                slice(columns.min(), columns.max() + 1),
            )
            prediction = prediction[region]
            reference = reference[region]
            valid = valid[region]

            for label in range(NUM_CLASSES):
                predicted = (prediction == label) & valid
                actual = (reference == label) & valid

                intersections[label] += (predicted & actual).sum()
                predicted_counts[label] += predicted.sum()
                reference_counts[label] += actual.sum()
                boundary_distances[label].append(
                    calculate_hd95(predicted, actual)
                )

results = []

for label in range(NUM_CLASSES):
    intersection = intersections[label]
    denominator = predicted_counts[label] + reference_counts[label]
    union = denominator - intersection

    distances = np.asarray(boundary_distances[label])
    finite_distances = distances[np.isfinite(distances)]

    results.append({
        "Label": label,
        "Structure": LABEL_NAMES[label],
        "Dice": 2 * intersection / denominator if denominator else np.nan,
        "IoU": intersection / union if union else np.nan,
        "Mean HD95, finite slices (pixels)": (
            finite_distances.mean() if finite_distances.size else np.nan
        ),
        "Finite HD95 slices": int(np.isfinite(distances).sum()),
        "One mask empty (HD95 infinite)": int(np.isinf(distances).sum()),
        "Both masks empty (HD95 undefined)": int(np.isnan(distances).sum()),
    })

metrics = pd.DataFrame(results).set_index("Label")
display(metrics.round(4))

print("Prostate results:")
display(metrics.loc[[PROSTATE_LABEL]].round(4))

metrics.to_csv(checkpoint_dir / "isensse_validation_metrics.csv")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Select the model and its best checkpoint.
# ---------------------------------------------------------

# BASELINE — use these settings in the baseline notebook:
MODEL_NAME = "baseline"
MODEL_CLASS = UNet
CHECKPOINT = project_dir / "checkpoints" / "baseline_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

# Load only your own trusted checkpoint.
checkpoint = torch.load(
    CHECKPOINT, map_location="cpu", weights_only=False
)

test_model = MODEL_CLASS(
    classes=checkpoint["num_classes"],
    base=checkpoint["base"],
)
test_model.load_state_dict(checkpoint["model_state_dict"])
test_model = test_model.to(device).eval()

num_classes = checkpoint["num_classes"]
target_size = tuple(checkpoint["target_size"])
prostate_label = checkpoint["prostate_label"]

print("Model:", MODEL_NAME)
print("Best checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint validation Dice:", checkpoint["val_prostate_dice"])

# Release the checkpoint's optimiser tensors.
del checkpoint

# ---------------------------------------------------------
# 2. Load the same held-out test split for every model.
# ---------------------------------------------------------

test_dataset = HipMRIDataset(
    splits["test"],
    target_size=target_size,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

label_names = {
    0: "Outside body",
    1: "Remaining tissue",
    2: "Bone",
    3: "Bladder",
    4: "Rectum",
    5: "Prostate",
}

print("Test slices:", len(test_dataset))

# ---------------------------------------------------------
# 3. Define the same HD95 calculation for every model.
# ---------------------------------------------------------

def test_hd95(predicted, reference):
    if not predicted.any() and not reference.any():
        return np.nan

    if not predicted.any() or not reference.any():
        return np.inf

    predicted_boundary = predicted & ~binary_erosion(
        predicted, border_value=0
    )
    reference_boundary = reference & ~binary_erosion(
        reference, border_value=0
    )

    distance_to_reference = distance_transform_edt(
        ~reference_boundary
    )
    distance_to_prediction = distance_transform_edt(
        ~predicted_boundary
    )

    distances = np.concatenate([
        distance_to_reference[predicted_boundary],
        distance_to_prediction[reference_boundary],
    ])
    return float(np.percentile(distances, 95))

# ---------------------------------------------------------
# 4. Evaluate predictions without updating model weights.
# ---------------------------------------------------------

intersections = np.zeros(num_classes, dtype=np.int64)
predicted_counts = np.zeros(num_classes, dtype=np.int64)
reference_counts = np.zeros(num_classes, dtype=np.int64)
slice_records = []
sample_index = 0

with torch.inference_mode():
    for images, masks in tqdm(
        test_loader, desc=f"Testing {MODEL_NAME}", mininterval=5
    ):
        predictions = (
            test_model(images.to(device))
            .argmax(dim=1)
            .cpu()
            .numpy()
        )
        masks = masks.numpy()

        for prediction, reference in zip(predictions, masks):
            image_path = Path(splits["test"][sample_index][0])
            sample_index += 1

            # Crop away the artificial padding before measuring boundaries.
            valid = reference != -100
            rows, columns = np.where(valid)
            region = (
                slice(rows.min(), rows.max() + 1),
                slice(columns.min(), columns.max() + 1),
            )
            prediction = prediction[region]
            reference = reference[region]
            valid = valid[region]

            for label in range(num_classes):
                predicted = (prediction == label) & valid
                actual = (reference == label) & valid

                intersection = int((predicted & actual).sum())
                predicted_count = int(predicted.sum())
                reference_count = int(actual.sum())

                intersections[label] += intersection
                predicted_counts[label] += predicted_count
                reference_counts[label] += reference_count

                denominator = predicted_count + reference_count
                union = denominator - intersection

                slice_records.append({
                    "model": MODEL_NAME,
                    "file": image_path.name,
                    "patient": image_path.name.split("_")[1],
                    "label": label,
                    "structure": label_names[label],
                    "dice": (
                        2 * intersection / denominator
                        if denominator else np.nan
                    ),
                    "iou": intersection / union if union else np.nan,
                    "hd95_pixels": test_hd95(predicted, actual),
                    "false_negative_pixels": int(
                        (actual & ~predicted).sum()
                    ),
                    "false_positive_pixels": int(
                        (predicted & ~actual).sum()
                    ),
                    "missed_structure": (
                        reference_count > 0 and predicted_count == 0
                    ),
                    "false_structure": (
                        reference_count == 0 and predicted_count > 0
                    ),
                    "both_empty": denominator == 0,
                })

# ---------------------------------------------------------
# 5. Summarise pooled overlap and per-slice boundary distances.
# ---------------------------------------------------------

slice_metrics = pd.DataFrame(slice_records)
summary = []

for label in range(num_classes):
    subset = slice_metrics[slice_metrics["label"] == label]
    distances = subset["hd95_pixels"].to_numpy()
    finite = distances[np.isfinite(distances)]

    denominator = predicted_counts[label] + reference_counts[label]
    union = denominator - intersections[label]

    summary.append({
        "Label": label,
        "Structure": label_names[label],
        "Dice": (
            2 * intersections[label] / denominator
            if denominator else np.nan
        ),
        "IoU": (
            intersections[label] / union if union else np.nan
        ),
        "Mean finite HD95 (pixels)": (
            finite.mean() if finite.size else np.nan
        ),
        "Finite HD95 slices": int(np.isfinite(distances).sum()),
        "Missed structure slices": int(subset["missed_structure"].sum()),
        "False structure slices": int(subset["false_structure"].sum()),
        "Both empty slices": int(subset["both_empty"].sum()),
    })

test_metrics = pd.DataFrame(summary).set_index("Label")

print(f"\n{MODEL_NAME}: held-out test results")
display(test_metrics.round(4))

print("\nProstate:")
display(test_metrics.loc[[prostate_label]].round(4))

# Save summaries and individual slices for later failure analysis.
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)

test_metrics.to_csv(results_dir / f"{MODEL_NAME}_test_metrics.csv")
slice_metrics.to_csv(
    results_dir / f"{MODEL_NAME}_test_slice_metrics.csv",
    index=False,
)

print("Saved results to:", results_dir)

In [ ]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version.split()[0])

for package in [
    "torch", "numpy", "nibabel", "pandas",
    "matplotlib", "scipy", "tqdm", "nbformat",
]:
    print(f"{package}=={metadata.version(package)}")

print("PyTorch CUDA version:", torch.version.cuda)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "CPU",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Read the results used to select common examples.
baseline = pd.read_csv(
    project_dir / "results" / "baseline_test_slice_metrics.csv"
)
isensee = pd.read_csv(
    project_dir / "results" / "isensee_test_slice_metrics.csv"
)

comparison = baseline[baseline["label"] == 5].merge(
    isensee[isensee["label"] == 5],
    on=["file", "patient", "label"],
    suffixes=("_baseline", "_isensee"),
    validate="one_to_one",
)
comparison["dice_change"] = (
    comparison["dice_isensee"] - comparison["dice_baseline"]
)

# Select a different candidate for each category.
# These rankings suggest examples; visual inspection determines the failure.
rankings = [
    (
        "Most missed tissue",
        comparison.sort_values(
            "false_negative_pixels_baseline", ascending=False
        ),
    ),
    (
        "Most incorrectly included tissue",
        comparison.sort_values(
            "false_positive_pixels_baseline", ascending=False
        ),
    ),
    (
        "Largest Isensee Dice improvement",
        comparison.dropna(subset=["dice_change"]).sort_values(
            "dice_change", ascending=False
        ),
    ),
    (
        "Largest Isensee Dice reduction",
        comparison.dropna(subset=["dice_change"]).sort_values(
            "dice_change"
        ),
    ),
]

selected, used = [], set()
for category, ranked in rankings:
    for filename in ranked["file"]:
        if filename not in used:
            selected.append((category, filename))
            used.add(filename)
            break

# Match the selected filenames to actual TEST images and masks.
test_pairs_by_name = {
    pair[0].name: pair for pair in splits["test"]
}
selected_pairs = [
    test_pairs_by_name[filename] for _, filename in selected
]

# Use the target size loaded with this model's checkpoint.
examples = HipMRIDataset(selected_pairs, target_size=target_size)

fig, axes = plt.subplots(
    len(selected), 4,
    figsize=(14, 3.8 * len(selected)),
    squeeze=False,
)

test_model.eval()

for row, (category, filename) in enumerate(selected):
    image, mask = examples[row]

    with torch.inference_mode():
        prediction = (
            test_model(image.unsqueeze(0).to(device))
            .argmax(dim=1)[0]
            .cpu().numpy()
        )

    image = image[0].numpy()
    mask = mask.numpy()

    # Crop away padding so it does not appear in the figures.
    valid = mask != -100
    rows, columns = np.where(valid)
    region = (
        slice(rows.min(), rows.max() + 1),
        slice(columns.min(), columns.max() + 1),
    )
    image = image[region]
    mask = mask[region]
    prediction = prediction[region]

    actual = mask == 5
    predicted = prediction == 5

    true_positive = actual & predicted
    false_positive = predicted & ~actual
    false_negative = actual & ~predicted

    # Show prostate only, keeping the MRI visible underneath.
    axes[row, 0].imshow(image, cmap="gray")

    for column, binary_mask in [(1, actual), (2, predicted)]:
        axes[row, column].imshow(image, cmap="gray")
        overlay = np.zeros((*binary_mask.shape, 4))
        overlay[binary_mask] = [1, 0.65, 0, 0.55]
        axes[row, column].imshow(overlay)

    # Green: correct overlap; red: extra tissue; blue: missed tissue.
    error_overlay = np.zeros((*actual.shape, 4))
    error_overlay[true_positive] = [0, 1, 0, 0.45]
    error_overlay[false_positive] = [1, 0, 0, 0.7]
    error_overlay[false_negative] = [0, 0.45, 1, 0.7]

    axes[row, 3].imshow(image, cmap="gray")
    axes[row, 3].imshow(error_overlay)

    for ax, title in zip(
        axes[row],
        ["MRI", "Reference prostate", "Predicted prostate", "Error overlay"],
    ):
        ax.set_title(title)
        ax.axis("off")

    # Identify the exact case and the reason it was selected.
    axes[row, 0].set_title(
        f"{category}\n{filename}", fontsize=9
    )

legend = [
    Patch(color="green", label="Correct overlap"),
    Patch(color="red", label="False positive: extra tissue"),
    Patch(color="dodgerblue", label="False negative: missed tissue"),
]
fig.legend(handles=legend, loc="lower center", ncol=3)
fig.suptitle(f"{MODEL_NAME}: candidate test failure cases", fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])

output_dir = project_dir / "results" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{MODEL_NAME}_failure_candidates.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", output_path)

In [ ]:
# Display per-slice prostate metrics for the visualised cases.
results_dir = project_dir / "results"

model_files = {
    "Baseline": "baseline_test_slice_metrics.csv",
    "Residual": "residual_test_slice_metrics.csv",
    "Isensee-inspired": "isensee_test_slice_metrics.csv",
}

case_results = []

for model_name, filename in model_files.items():
    csv_path = results_dir / filename

    if not csv_path.exists():
        print(f"Skipping {model_name}: {filename} not found.")
        continue

    frame = pd.read_csv(csv_path)
    prostate = frame[frame["label"] == 5]

    for category, case_filename in selected:
        matching = prostate[prostate["file"] == case_filename]

        if matching.empty:
            print(f"No result for {model_name}: {case_filename}")
            continue

        result = matching.iloc[0]

        case_results.append({
            "Selection": category,
            "Slice": case_filename,
            "Model": model_name,
            "Dice": result["dice"],
            "IoU": result["iou"],
            "HD95 (pixels)": result["hd95_pixels"],
            "Missed pixels": result["false_negative_pixels"],
            "Extra pixels": result["false_positive_pixels"],
        })

case_metrics = pd.DataFrame(case_results)

for category, case_filename in selected:
    print(f"\n{category}: {case_filename}")

    display(
        case_metrics[case_metrics["Slice"] == case_filename]
        .set_index("Model")[
            ["Dice", "IoU", "HD95 (pixels)", "Missed pixels", "Extra pixels"]
        ]
        .round(4)
    )

case_metrics.to_csv(
    results_dir / "selected_failure_case_metrics.csv",
    index=False,
)

In [ ]:
model_parameters = filter(lambda p: p.requires_grad, test_model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
print(params)